In [1]:
import heapq


class Solution:
  def busiestServers(self, k: int, arrival: list[int], load: list[int]) -> list[int]:
    size = 1
    while size < k:
      size *= 2
    tree = [0] * (2 * size)
    for i in range(k):
      tree[size + i] = 1
    for i in range(size - 1, 0, -1):
      tree[i] = tree[2 * i] + tree[2 * i + 1]

    def update(idx, value):
      idx += size
      tree[idx] = value
      idx //= 2
      while idx:
        tree[idx] = tree[2 * idx] + tree[2 * idx + 1]
        idx //= 2

    def find(start):
      def search(node, left, right):
        if right <= start or tree[node] == 0:
          return -1
        if right - left == 1:
          return left
        mid = (left + right) // 2
        result = search(node * 2, left, mid)
        if result != -1:
          return result
        return search(node * 2 + 1, mid, right)
      return search(1, 0, size)

    busy = []
    count = [0] * k
    for i in range(len(arrival)):
      current_time = arrival[i]
      while busy and busy[0][0] <= current_time:
        _, server = heapq.heappop(busy)
        update(server, 1)
      start = i % k
      server = find(start)
      if server == -1:
        server = find(0)
      if server == -1:
        continue
      update(server, 0)
      count[server] += 1
      heapq.heappush(
        busy,
        (current_time + load[i], server)
      )
    max_count = max(count)
    return [
      server
      for server in range(k)
      if count[server] == max_count
    ]

In [2]:
k = 3
arrival = [1, 2, 3, 4, 5]
load = [5, 2, 3, 3, 3] 
Solution().busiestServers(k=k, arrival=arrival, load=load)

[1]

In [3]:
k = 3
arrival = [1, 2, 3, 4]
load = [1, 2, 1, 2]
Solution().busiestServers(k=k, arrival=arrival, load=load)

[0]

In [4]:
k = 3
arrival = [1, 2, 3]
load = [10, 12, 11]
Solution().busiestServers(k=k, arrival=arrival, load=load)

[0, 1, 2]